In [24]:
import os
from tqdm import tqdm
from pathlib import Path

# Log only Tensorflow critical errors
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

from sequence_utils import (
    extract_region,
    get_hg38, get_human_record_ids, get_human_basenji_regions,
    get_mm10, get_mouse_record_ids, get_mouse_basenji_regions,
    MOUSE_REF_FOLDER, MOUSE_TFR_FOLDER,
    get_coords_from_blast,
    get_sequences_for_record,
    expand_regions,
)

In [ ]:
from functools import partial

extract_mouse_region = partial(extract_region, seqs_per_chr=(mm10_per_chr := get_mm10()))
extract_human_region = partial(extract_region, seqs_per_chr=(hg38_per_chr := get_hg38()))

expand_regions_to_3x = partial(expand_regions, to_left=1+(SEQLEN:=131072), to_right=0+SEQLEN)

66it [00:17,  3.69it/s]
23it [00:17,  1.32it/s]


In [13]:
mouse_seqs_set = mouse_seqs_df.apply(extract_mouse_region, axis=1).to_list()

In [9]:
display(human_seqs_df := get_human_basenji_regions())
display(mouse_seqs_df := get_mouse_basenji_regions())

,chromosome,start,end,subset
0,chr18,928386,1059458,train
1,chr4,113630947,113762019,train
2,chr11,18427720,18558792,train
3,chr16,85805681,85936753,train
4,chr3,158386188,158517260,train
...,...,...,...,...
38166,chr19,33204702,33335774,test
38167,chr14,41861379,41992451,test
38168,chr19,30681544,30812616,test
38169,chr14,61473198,61604270,test


,chromosome,start,end,subset
0,chr4,34106647,34237719,train
1,chr5,52207747,52338819,train
2,chr19,20136862,20267934,train
3,chr14,61845439,61976511,train
4,chr15,6592346,6723418,train
...,...,...,...,...
33516,chr2,110707432,110838504,test
33517,chr12,59936948,60068020,test
33518,chrX,56644074,56775146,test
33519,chrX,23595900,23726972,test


In [19]:
record_id = (mouse_record_ids := get_mouse_record_ids())[(WHICH_RECORD := 6)]
print(f"{record_id=}")

record_id='test-1-7'


In [20]:
sequences_from_tfr  = get_sequences_for_record(record_id)

225it [00:00, 1597.94it/s]


In [ ]:
blast_results_df    = get_coords_from_blast(record_id)
expanded_regions_df = expand_regions_to_3x(blast_results_df)
sequences_from_ref  = expanded_regions_df.apply(extract_region, axis=1, seqs_per_chr=mm10_per_chr, expected_length=3*SEQLEN)

assert all([ v == sequences_from_ref.iloc[i][SEQLEN:-SEQLEN] for i, (k, v) in enumerate(sequences_from_tfr.items()) ])